# Week 3, day 1 (afternoon) — Extra practice 01 SOLUTIONS: Series   (L01)

Executed in the lab image against the real `../data/sales.csv`. Every quoted
number is what it actually printed.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Extra practice 01 — Series. Run this once.
import pandas as pd

sales = pd.read_csv("../data/sales.csv")
profit = sales["Profit"]
region = sales["Region"]

print("rows:", len(sales))
print(profit.head())

### Question 1

`dtype float64`, `name Profit`, `type Series`.

A single column pulled out of a DataFrame is a Series, and it remembers
which column it came from in `.name`. That name is what becomes the column
heading if you assign it back into a frame.

In [ ]:
print("dtype:", profit.dtype)
print("name: ", profit.name)
print("type: ", type(profit).__name__)
print()
print(profit.head(3))

### Question 2

sum `34833.49`, mean `116.11`, min `-1111.45`, max `3030.16`.

The minimum is negative, which is the finding. A column called `Profit`
that has negative values is doing two jobs — profit and loss — and any
summary that assumes it is non-negative is wrong.

In [ ]:
print("sum: ", round(profit.sum(), 2))
print("mean:", round(profit.mean(), 2))
print("min: ", round(profit.min(), 2))
print("max: ", round(profit.max(), 2))

### Question 3

**128 of 300** orders lose money — `42.7%`.

Nearly half the orders in this file are loss-making. That is a fact about
the business, not about the data, and it is invisible in the mean: `116.11`
per order sounds like a healthy margin.

This is why `.describe()` and `.min()` matter before any averaging. A mean
over a mixture of gains and losses tells you the net, and hides that the
distribution has two very different populations in it.

In [ ]:
losses = profit < 0
print("loss-making orders:", losses.sum(), "of", len(profit))
print("percentage: %.1f%%" % (100 * losses.sum() / len(profit)))

### Question 4

losses `-11358.07`, gains `46191.56`, sum `34833.49` — matching `profit.sum()` when both are rounded.

The two halves reconcile. Note the comparison was made on rounded values,
deliberately — worksheet 12's lesson. The raw floats may differ in the last
bits because they were added in different groupings, and `==` on the
unrounded numbers is a test you have no reason to expect to pass.

The useful shape here is that a £46k gain and an £11k loss net to £35k. The
net is real, and so is the fact that a quarter of the gross was given back.

In [ ]:
neg = profit[profit < 0].sum()
pos = profit[profit >= 0].sum()
print("losses: ", round(neg, 2))
print("gains:  ", round(pos, 2))
print("sum:    ", round(neg + pos, 2))
print("overall:", round(profit.sum(), 2))
print("agree when rounded:", round(neg + pos, 2) == round(profit.sum(), 2))

### Question 5

`dtype str`, `8` distinct regions. -> `value_counts()` is ordered by frequency; `unique()` is ordered by **first appearance**.

Two different orderings and neither is alphabetical. `value_counts()` sorts
by count descending. `unique()` returns values in the order it first met
them while scanning the column — so its output depends on the row order of
your file and will change if the file is re-sorted.

Never rely on `unique()` order for anything a human will read. Use
`sorted(...)` if you want it stable.

And `Prarie` is there again, second by volume — 75 of 300 rows sit under a
misspelling.

In [ ]:
print("dtype:   ", region.dtype)
print("distinct:", region.nunique())
print()
print(region.value_counts())
print()
print("unique() order:", list(region.unique()))

### Question 6

`groupby("Region")["Sales"].sum()` -> a Series **indexed by region name**. -> `Yukon` is `5307.05`.

This is the first Series in the class whose index carries real meaning
rather than being row numbers. `by_region["Yukon"]` is a label lookup that
reads like English.

Note `groupby` returned the regions in **alphabetical** order, unlike either
ordering in Q5. That is a third convention in the same sheet, which is the
reason to sort explicitly whenever the order matters.

In [ ]:
by_region = sales.groupby("Region")["Sales"].sum()
print(by_region)
print()
print("index:", list(by_region.index))
print("Yukon:", round(by_region["Yukon"], 2))

### Question 7

Top three: `Prarie 56716.70`, `West 56072.55`, `Atlantic 48695.54`. -> `161484.79` of `236825.0` — **68.2%**.

Three of eight regions account for over two thirds of revenue. That is the
kind of concentration worth knowing before you plan anything per-region:
the five smallest regions together are less than a third of the business.

The top two are within 650 of each other, on totals of about 56,000 — a
difference of roughly 1%. Reporting 'Prarie is our biggest region' is true
and fragile; one large West order would reverse it.

In [ ]:
by_region = sales.groupby("Region")["Sales"].sum().sort_values(ascending=False)
print(by_region.head(3))
print()
top3 = by_region.head(3).sum()
print("top 3 total:", round(top3, 2))
print("all regions:", round(by_region.sum(), 2))
print("share: %.1f%%" % (100 * top3 / by_region.sum()))

### Question 8

`by_region["Prairie"]` -> **raises** `KeyError: 'Prairie'`. -> what is actually there is `['Prarie']`.

The correctly-spelled English word is the one that fails. The data says
`Prarie`, so `Prairie` matches nothing.

Here the lookup raises, which is the good case — you find out immediately.
The dangerous version is the same typo inside a **filter**:
`sales[sales["Region"] == "Prairie"]` returns an empty frame, no error, and
every downstream aggregate quietly reports zero for a region worth £56,000.

Which is why Q5's `value_counts()` is not an optional first look. Read the
values that are actually in the column before you write a condition against
them.

In [ ]:
by_region = sales.groupby("Region")["Sales"].sum()
print("what is actually there:", [r for r in by_region.index if r.startswith("Pra")])
print(by_region["Prairie"])